# Experimentación con Parámetros del Modelo


## Configuración de variables y modelo

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Configuración del cliente para Endpoint de Proyecto en AI Foundry
client = OpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT")
)

def ask_params(prompt, temp=1.0, top_p=1.0, pres=0.0, freq=0.0):
    deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
    response = client.chat.completions.create(
        model=deployment_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        top_p=top_p,
        presence_penalty=pres,
        frequency_penalty=freq
    )
    return response.choices[0].message.content

## 2.1 Experimentación con Temperature
Usaremos un prompt creativo para observar cómo varía la aleatoriedad.

In [ ]:
prompt_temp = "Escribe una frase para animar a un enfermo terminal."
temperaturas = [0.0, 0.5, 1.0, 1.5]

for t in temperaturas:
    print(f"--- Temperatura: {t} ---")
    print(ask_params(prompt_temp, temp=t))
    print("\n")

--- Temperatura: 0.0 ---
"Tu fortaleza y tu espíritu son un ejemplo para todos; cada momento que compartimos contigo es un regalo invaluable lleno de amor y esperanza."


--- Temperatura: 0.5 ---
"Tu valentía y fortaleza inspiran a todos los que te rodean; cada día es un regalo, y en este momento, tu amor y presencia dejan huellas que nunca se borrarán."


--- Temperatura: 1.0 ---
"Tu fortaleza y valentía brillan más que cualquier oscuridad; cada momento compartido contigo es un regalo para quienes te queremos."


--- Temperatura: 1.5 ---
Aunque el camino sea incierto, cada día te rodea amor y estás dejando huellas imborrables en quienes te quieren. Tu fortaleza y serenidad son un ejemplo para todos; vive hoy con alegría y rodeado de esa luz que inspiras.




### Análisis comparativo

0.0 (Determinista): La respuesta es rígida y usa frases muy comunes. El modelo no arriesga y elige siempre la opción más probable.

0.5 (Equilibrado): Se mantiene la estructura, pero el lenguaje es algo más fluido y menos robótico.

1.0 (Creativo): El tono es más natural y humano. Introduce metáforas (como "oscuridad") y varía la construcción de las frases.

1.5 (Aleatorio): El modelo rompe la estructura lineal. Es mucho más descriptivo y largo, explorando ideas nuevas que no aparecían en los niveles bajos.

## 2.2 Experimentación con Top_P


In [5]:
prompt_top_p = "Dime una cualidad de un buen líder."
valores_p = [0.1, 0.5, 0.9, 1.0]

for p in valores_p:
    print(f"--- Top_P: {p} (Temp=1.0) ---")
    print(ask_params(prompt_top_p, temp=1.0, top_p=p))
    print("\n")

--- Top_P: 0.1 (Temp=1.0) ---
Una cualidad esencial de un buen líder es la **empatía**. La capacidad de comprender y conectar con las emociones, necesidades y perspectivas de los demás permite al líder construir relaciones sólidas, fomentar la confianza y motivar a su equipo de manera efectiva.


--- Top_P: 0.5 (Temp=1.0) ---
Una cualidad fundamental de un buen líder es la **empatía**. La capacidad de comprender y conectar con las emociones, necesidades y perspectivas de su equipo permite al líder fomentar un ambiente de confianza, colaboración y motivación. La empatía ayuda a construir relaciones sólidas y a tomar decisiones que beneficien tanto a las personas como a los objetivos organizacionales.


--- Top_P: 0.9 (Temp=1.0) ---
Una cualidad fundamental de un buen líder es la **empatía**. La empatía permite al líder comprender y valorar las emociones, necesidades y perspectivas de su equipo, fomentando una conexión genuina y un ambiente de confianza y colaboración. Esto no solo mejor

### Análisis comparativo

Top_P 0.1: El modelo solo considera el 10% de las palabras más probables. El resultado es muy "de manual" y predecible, similar a una temperatura baja.

Top_P 0.5: La respuesta es moderada. Se nota que el modelo descarta las opciones más raras pero no se limita a la más obvia.

Top_P 0.9 - 1.0: El modelo tiene a su disposición casi todo el vocabulario. La diferencia con la temperatura es que, incluso con un Top_P de 1.0, si la temperatura es moderada, el modelo se mantiene coherente porque no está "forzando" las palabras improbables, sino que simplemente no tiene prohibido usarlas.

## 2.3 Experimentación con Penalties
Probaremos un prompt que invite a la redundancia.

In [7]:
prompt_rep = "Describe las ventajas de comer fruta de forma detallada, usando frases largas."

print("Sin Penalizaciones \n")
print(ask_params(prompt_rep, pres=0.0, freq=0.0))

print("\nCon Presence Penalty (0.6) \n")
print(ask_params(prompt_rep, pres=0.6, freq=0.0))

print("\nCon Frequency Penalty (0.8) \n")
print(ask_params(prompt_rep, pres=0.0, freq=0.8))

Sin Penalizaciones 

Consumir frutas de manera regular aporta innumerables beneficios para la salud debido a que son alimentos naturales, ricos en nutrientes esenciales y bajas en calorías, convirtiéndose en una de las mejores elecciones para mantener un estilo de vida equilibrado. En primer lugar, las frutas son una fuente importante de vitaminas como la vitamina C, que fortalece el sistema inmunológico, mejora la producción de colágeno y actúa como antioxidante para combatir los radicales libres, ayudando a retrasar el envejecimiento celular. Asimismo, muchas frutas aportan vitaminas del grupo B, esenciales para mantener el buen funcionamiento del sistema nervioso y para el metabolismo energético, así como vitamina A, que contribuye a la salud visual y al buen estado de la piel.

Además, las frutas son ricas en minerales como el potasio, clave para regular el equilibrio de líquidos en el cuerpo, mantener una presión arterial saludable y apoyar el rendimiento muscular y cardíaco. Tamb

### Análisis comparativo

Sin Penalizaciones (0.0): El texto es muy extenso y técnico, pero tiende a ser redundante. Repite constantemente estructuras como "las frutas son...", "también contienen..." y abusa de términos como "salud" y "cuerpo".

Presence Penalty (0.6): El modelo hace un esfuerzo por introducir temas nuevos. A diferencia del primer texto, aquí añade una dimensión emocional y mental, mencionando el "triptófano" y la "serotonina" (la hormona de la felicidad). Esta penalización "empuja" al modelo a explorar conceptos que aún no han aparecido.

Frequency Penalty (0.8): El impacto es notable en el vocabulario. El modelo evita repetir las mismas palabras exactas.

Diferencia clave observada:

Presence Penalty evitó que el modelo se quedara estancado solo en la parte nutricional, obligándolo a hablar de salud mental.

Frequency Penalty evitó que el texto sonara monótono, obligando al modelo a usar sinónimos y conceptos más específicos para no repetir los términos "fruta" o "beneficios" constantemente.

## 2.4 Preguntas Teóricas

**¿Cuál es la diferencia entre top_p y temperature?**
La temperatura afecta a la probabilidad de todos los tokens por igual, 'aplanando' la distribución para que los menos probables tengan oportunidad. Top_p, en cambio, corta el pastel: solo deja elegir entre el grupo de palabras que suman ese porcentaje de probabilidad, ignorando el resto.

**¿Por qué no se recomienda ajustar top_p y temperature al mismo tiempo?**
Porque ambos buscan controlar la aleatoriedad. Si ajustas ambos de forma agresiva, se solapan y el control se vuelve caótico o demasiado restrictivo, dificultando saber qué parámetro está causando qué efecto en la respuesta.

**¿Cuál es la diferencia entre presence_penalty y frequency_penalty?**
Presence penalty castiga a un token por el simple hecho de haber aparecido una vez, para forzar al modelo a cambiar de tema. Frequency penalty castiga al token según cuántas veces ha salido; cuantas más veces se repite una palabra, más difícil le resulta al modelo volver a usarla.

## Conclusiones finales

Para tareas de código o extracción de datos, lo mejor es usar `temperature: 0` para asegurar precisión. Para tareas creativas, una temperatura entre 0.7 y 0.9 combinada con penalties bajos ayuda a que el texto sea fluido y variado. Si el modelo se vuelve repetitivo en descripciones largas, subir `frequency_penalty` es la solución más directa.